# Univariate Questions\n\nSingle-variable rankings and distributions on the cleaned fight-level dataset (`data/master.csv` via `src/data/cleaning.py`).\n\nQuestions answered here:\n1. Who has fought in the most title fights?\n2. Who has the most career wins? Most losses?\n3. Who has the longest career in the sport?\n4. Which event had the most KO/TKOs, submissions, decisions?\n5. How does fighter age vary across weight divisions?"

In [ ]:
import sys
sys.path.insert(0, "../src/data")
sys.path.insert(0, "../src/features")

import pandas as pd
import matplotlib.pyplot as plt

from cleaning import clean_fights
from engineering import add_matchup_features

master = pd.read_csv("../data/master.csv")
fights = clean_fights(master)
fights = add_matchup_features(fights)  # needed for r_age/b_age in question 5

fights.shape

## 1. Most Title Fights\n\nEach fighter appears as either the red or blue corner per fight, so counting title fights per fighter needs both sides combined first."

In [ ]:
r_side = fights[["r_fighter_id", "r_fighter_name", "title_fight"]].rename(
    columns={"r_fighter_id": "fighter_id", "r_fighter_name": "fighter_name"})
b_side = fights[["b_fighter_id", "b_fighter_name", "title_fight"]].rename(
    columns={"b_fighter_id": "fighter_id", "b_fighter_name": "fighter_name"})
long = pd.concat([r_side, b_side])

title_counts = long[long["title_fight"]].groupby(["fighter_id", "fighter_name"]).size()
top_title = title_counts.sort_values(ascending=False).head(15).reset_index(name="title_fights")

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_title["fighter_name"], top_title["title_fights"])
ax.invert_yaxis()
ax.set_xlabel("title fights")
ax.set_title("Most Title Fights")
plt.tight_layout()
plt.show()

top_title

## 2. Most Wins, Most Losses\n\n`clean_fights` already dropped draws/no-contests (no winner_id), so every remaining row is a decisive result - each fighter's side is either a win or a loss, nothing else to account for."

In [ ]:
r_side = fights[["r_fighter_id", "r_fighter_name", "winner_id"]].rename(
    columns={"r_fighter_id": "fighter_id", "r_fighter_name": "fighter_name"})
b_side = fights[["b_fighter_id", "b_fighter_name", "winner_id"]].rename(
    columns={"b_fighter_id": "fighter_id", "b_fighter_name": "fighter_name"})
long = pd.concat([r_side, b_side])
long["is_win"] = (long["fighter_id"] == long["winner_id"]).astype(int)

wins = long[long["is_win"] == 1].groupby(["fighter_id", "fighter_name"]).size()
wins = wins.sort_values(ascending=False).head(15).reset_index(name="wins")

losses = long[long["is_win"] == 0].groupby(["fighter_id", "fighter_name"]).size()
losses = losses.sort_values(ascending=False).head(15).reset_index(name="losses")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(wins["fighter_name"], wins["wins"], color="tab:green")
axes[0].invert_yaxis()
axes[0].set_title("Most Wins")
axes[1].barh(losses["fighter_name"], losses["losses"], color="tab:red")
axes[1].invert_yaxis()
axes[1].set_title("Most Losses")
plt.tight_layout()
plt.show()

wins, losses

## 3. Longest Careers\n\nCareer span = time between a fighter's first and last fight in the dataset. Fighters with only 1 fight are excluded - a single fight has no span, and including them would put a wall of ties at 0 years at the bottom rather than being a meaningful comparison."

In [ ]:
r_side = fights[["r_fighter_id", "r_fighter_name", "event_date"]].rename(
    columns={"r_fighter_id": "fighter_id", "r_fighter_name": "fighter_name"})
b_side = fights[["b_fighter_id", "b_fighter_name", "event_date"]].rename(
    columns={"b_fighter_id": "fighter_id", "b_fighter_name": "fighter_name"})
long = pd.concat([r_side, b_side])

career = long.groupby(["fighter_id", "fighter_name"]).agg(
    first_fight=("event_date", "min"), last_fight=("event_date", "max"), n_fights=("event_date", "size"))
career = career[career["n_fights"] >= 2]
career["career_years"] = (career["last_fight"] - career["first_fight"]).dt.days / 365.25
career = career.sort_values("career_years", ascending=False).head(15).reset_index()

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(career["fighter_name"], career["career_years"], color="tab:purple")
ax.invert_yaxis()
ax.set_xlabel("career span (years)")
ax.set_title("Longest Careers")
plt.tight_layout()
plt.show()

career

## 4. Event With the Most KO/TKOs, Submissions, Decisions\n\n`method` has 11 raw values (`Decision - Unanimous`, `Decision - Split`, `Decision - Majority`, `Decision` all mean the same outcome type). Grouping into `KO/TKO`, `Submission`, `Decision`, `Other` before counting per event - otherwise a card with a mix of decision types would be undercounted against one with a single decision label."

In [ ]:
method = fights["method"].astype(str)
fights["method_group"] = "Other"
fights.loc[method == "KO/TKO", "method_group"] = "KO/TKO"
fights.loc[method == "Submission", "method_group"] = "Submission"
fights.loc[method.str.startswith("Decision"), "method_group"] = "Decision"

event_method = fights.groupby(["event_name", "method_group"]).size().reset_index(name="count")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = {"KO/TKO": "tab:orange", "Submission": "tab:blue", "Decision": "tab:gray"}
for ax, method_name in zip(axes, ["KO/TKO", "Submission", "Decision"]):
    top = event_method[event_method["method_group"] == method_name].sort_values("count", ascending=False).head(10)
    ax.barh(top["event_name"], top["count"], color=colors[method_name])
    ax.invert_yaxis()
    ax.set_title(f"Most {method_name}")
plt.tight_layout()
plt.show()

## 5. Age Distribution by Division\n\n`weight_class` has 154 raw values in the source data - most are one-off tournament labels (e.g. \"Ultimate Fighter 13 Welterweight Tournament\"). Restricting to the actual standing UFC divisions keeps the chart readable; `\"Bout\"` (54 fights, an early-UFC scraping artifact rather than a real division) is excluded too."

In [ ]:
division_order = ["Women's Strawweight", "Women's Flyweight", "Women's Bantamweight",
                   "Flyweight", "Bantamweight", "Featherweight", "Lightweight",
                   "Welterweight", "Middleweight", "Light Heavyweight", "Heavyweight",
                   "Catch Weight", "Open Weight"]

r_age = fights[["r_age", "weight_class"]].rename(columns={"r_age": "age"})
b_age = fights[["b_age", "weight_class"]].rename(columns={"b_age": "age"})
age_long = pd.concat([r_age, b_age]).dropna()
age_long = age_long[age_long["weight_class"].isin(division_order)]

data = [age_long.loc[age_long["weight_class"] == division, "age"] for division in division_order]

fig, ax = plt.subplots(figsize=(12, 6))
ax.boxplot(data, tick_labels=division_order)
ax.set_xticklabels(division_order, rotation=45, ha="right")
ax.set_ylabel("age at time of fight")
ax.set_title("Age Distribution by Division")
plt.tight_layout()
plt.show()

age_long.groupby("weight_class")["age"].median().reindex(division_order)